In [44]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
 
from sklearn.preprocessing import PowerTransformer


In [45]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [46]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [47]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

## Test dataset: MAASTRO 

In [48]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [49]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [50]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [51]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [52]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [53]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [54]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# Set y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [55]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [56]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [57]:
# Change the name of a column 'DFS_event' in the clincial_test 
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [58]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

## Feature Selection

### COX PLSR 

In [59]:
# Choose features from the result of Cox PLSR in R
selected_features = [
"first_order_Maximum_CT",
"LBP_201_PET",
"LBP_003_PET",
"LBP_102_PET",
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16",
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2",
] 


In [60]:
X_plsr = X.loc[:, selected_features]
X_new = X_plsr.copy()

In [61]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Yeo-Johnson Transformation

In [62]:
# Copy the original X for later 
original_X = X.copy()

In [63]:
# Transform X_new 
# Set the categorical_columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Apply Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
X_new_numeric_transformed = pt.fit_transform(X_new_numeric)

# Create DataFrame with transformed numerical data
X_new_numeric_transformed = pd.DataFrame(X_new_numeric_transformed, 
                                         columns=X_new_numeric.columns, 
                                         index=X_new.index)

# Concatenate transformed numerical data with categorical data
X_new_std = pd.concat([X_new_numeric_transformed, X_new_categoric], axis=1)

# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the transformation for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [64]:
X_new

,first_order_Maximum_CT,LBP_201_PET,LBP_003_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,1763.283818,0.000062,0.000123,0.000000,0.000416,0.000959
1,1432.109636,0.000349,0.000349,0.000000,0.001753,0.002776
2,1685.931373,0.000000,0.000000,0.000034,0.000230,0.001179
3,1329.347515,0.000000,0.000300,0.000000,0.000711,0.002748
4,1197.225910,0.000399,0.000000,0.000199,0.001033,0.002309
...,...,...,...,...,...,...
134,1267.959716,0.000000,0.000000,0.000000,0.000226,0.001347
135,1788.278096,0.000079,0.000039,0.000000,0.000183,0.000850
136,1478.297861,0.000000,0.000000,0.000000,0.000621,0.001111
137,1207.068493,0.000054,0.000000,0.000000,0.000265,0.000935


In [65]:
X_new_std

,first_order_Maximum_CT,LBP_201_PET,LBP_003_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,1.022943,-0.074611,1.042615,-0.784694,-0.128003,-0.456825
1,0.072727,1.689445,1.929439,-0.784694,1.761655,1.829690
2,0.853782,-1.065926,-0.799521,0.236612,-0.861897,0.033412
3,-0.390532,-1.065926,1.846691,-0.784694,0.668885,1.814149
4,-1.187658,1.782787,-0.799521,1.784568,1.208077,1.522029
...,...,...,...,...,...,...
134,-0.727488,-1.065926,-0.799521,-0.784694,-0.877225,0.352487
135,1.072604,0.138716,-0.000653,-0.784694,-1.082845,-0.735470
136,0.247868,-1.065926,-0.799521,-0.784694,0.464990,-0.108155
137,-1.118344,-0.175482,-0.799521,-0.784694,-0.706059,-0.515193


In [66]:
MAASTRO_new 

,first_order_Maximum_CT,LBP_201_PET,LBP_003_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,1226.795917,0.000026,0.000026,0.000026,0.000241,0.000768
1,1876.570382,0.000167,0.000056,0.000167,0.000621,0.001213
2,1269.217095,0.000000,0.000286,0.000057,0.000327,0.001176
3,1951.924395,0.000080,0.000160,0.000000,0.001419,0.001192
4,1646.853670,0.000035,0.000071,0.000000,0.000165,0.000766
...,...,...,...,...,...,...
94,2311.845731,0.000078,0.000000,0.000000,0.000739,0.001137
95,1299.857157,0.000054,0.000109,0.000000,0.001130,0.000962
96,1256.024949,0.000028,0.000000,0.000000,0.000226,0.000621
97,1668.742962,0.000000,0.000114,0.000000,0.000296,0.000947


In [67]:
MAASTRO_new_std

,first_order_Maximum_CT,LBP_201_PET,LBP_003_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,-0.984802,-0.597387,-0.238970,0.033184,-0.811666,-0.960377
1,1.231237,0.946684,0.263801,1.682719,0.464229,0.101598
2,-0.720051,-1.065926,1.817599,0.693043,-0.453072,0.027699
3,1.348682,0.152800,1.314752,-0.784694,1.579512,0.060650
4,0.758335,-0.453142,0.471935,-0.784694,-1.171859,-0.967045
...,...,...,...,...,...,...
94,1.750433,0.126817,-0.799521,-0.784694,0.727222,-0.052777
95,-0.545966,-0.173978,0.907497,-0.784694,1.322893,-0.450240
96,-0.799285,-0.565019,-0.799521,-0.784694,-0.877027,-1.409205
97,0.812692,-1.065926,0.962590,-0.784694,-0.577241,-0.485449


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [68]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 20:45:26,676] A new study created in memory with name: no-name-5847f1d3-6746-42dd-ab6f-47146e9f256e
python(32902) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.6550387596899225
Fold 3 C-index: 0.7872340425531915


[I 2024-04-16 20:45:37,829] A new study created in memory with name: no-name-100d6f6d-27a0-417f-b949-2f25b65f0198


Fold 4 C-index: 0.39923954372623577
Fold 5 C-index: 0.5450643776824035
[I 2024-04-16 20:45:37,806] Trial 0 finished with value: 0.5801041893518646 and parameters: {}. Best is trial 0 with value: 0.5801041893518646.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5801041893518646], datetime_start=datetime.datetime(2024, 4, 16, 20, 45, 27, 120085), datetime_complete=datetime.datetime(2024, 4, 16, 20, 45, 37, 804911), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5801041893518646


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2631227485062865
Fold 2 IBS: 0.21992045469997798
Fold 3 IBS: 0.18351218320743473
Fold 4 IBS: 0.3295941082344743
Fold 5 IBS: 0.25161822070737033
[I 2024-04-16 20:45:38,269] Trial 0 finished with value: 0.2495535430711088 and parameters: {}. Best is trial 0 with value: 0.2495535430711088.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2495535430711088], datetime_start=datetime.datetime(2024, 4, 16, 20, 45, 37, 870561), datetime_complete=datetime.datetime(2024, 4, 16, 20, 45, 38, 269409), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2495535430711088


In [69]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [70]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.58
train_ibs:  0.25


#### Test

In [71]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [72]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.553
IBS score: 0.246


In [73]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [74]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [75]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:45:38,592] A new study created in memory with name: no-name-f507e689-7430-41a3-8bfe-189fe30a1bf6


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-16 20:45:38,942] A new study created in memory with name: no-name-cedd67f3-44e1-4790-9e2b-ee32df04346d


Fold 1 C-index: 0.48804780876494025
Fold 2 C-index: 0.6918604651162791
Fold 3 C-index: 0.8170212765957446
Fold 4 C-index: 0.4144486692015209
Fold 5 C-index: 0.5128755364806867
[I 2024-04-16 20:45:38,931] Trial 0 finished with value: 0.5848507512318343 and parameters: {}. Best is trial 0 with value: 0.5848507512318343.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5848507512318343], datetime_start=datetime.datetime(2024, 4, 16, 20, 45, 38, 625095), datetime_complete=datetime.datetime(2024, 4, 16, 20, 45, 38, 931292), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5848507512318343


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724710126201854
Fold 2 IBS: 0.23203988007163143
Fold 3 IBS: 0.22898185146634253
Fold 4 IBS: 0.2419747781639664
Fold 5 IBS: 0.22939559565695583
[I 2024-04-16 20:45:39,346] Trial 0 finished with value: 0.23592784132418293 and parameters: {}. Best is trial 0 with value: 0.23592784132418293.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784132418293], datetime_start=datetime.datetime(2024, 4, 16, 20, 45, 38, 980129), datetime_complete=datetime.datetime(2024, 4, 16, 20, 45, 39, 344932), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784132418293


In [76]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [77]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.585
train_ibs:  0.236


#### Test

In [78]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [79]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.569


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [80]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [81]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:45:39,956] A new study created in memory with name: no-name-f535c884-5132-48e0-bd08-65a6fa32c5a6


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:45:40,367] Trial 0 finished with value: 0.5935639212572479 and parameters: {}. Best is trial 0 with value: 0.5935639212572479.


[I 2024-04-16 20:45:40,406] A new study created in memory with name: no-name-f1800494-6944-4373-a817-bb91eb4c929c




* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5935639212572479], datetime_start=datetime.datetime(2024, 4, 16, 20, 45, 40, 686), datetime_complete=datetime.datetime(2024, 4, 16, 20, 45, 40, 367287), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5935639212572479


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.262979850327254
Fold 2 IBS: 0.22664749640228837
Fold 3 IBS: 0.21369285752862316
Fold 4 IBS: 0.32792704532765343
Fold 5 IBS: 0.2513898907749928
[I 2024-04-16 20:45:40,996] Trial 0 finished with value: 0.25652742807216233 and parameters: {}. Best is trial 0 with value: 0.25652742807216233.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.25652742807216233], datetime_start=datetime.datetime(2024, 4, 16, 20, 45, 40, 517372), datetime_complete=datetime.datetime(2024, 4, 16, 20, 45, 40, 996707), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.25652742807216233


In [82]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [83]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.594
train_ibs:  0.257


#### Test

In [84]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [85]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.553


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.246


In [86]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [87]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:45:41,487] A new study created in memory with name: no-name-33e8053e-745c-4efc-af15-31590b2fca8e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:45:41,927] Trial 0 finished with value: 0.5935639212572479 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.5935639212572479.
Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:45:42,709] Trial 1 finished with value: 0.5935639212572479 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.5935639212572479.
Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:45:43,300] Trial 2 finished with value: 0.5935639212572479 and parameters: {'l1_ratio': 0.2269287684188466

Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:45:52,483] Trial 24 finished with value: 0.5927887274587983 and parameters: {'l1_ratio': 0.15433706302991543}. Best is trial 0 with value: 0.5935639212572479.
Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:45:52,790] Trial 25 finished with value: 0.5935639212572479 and parameters: {'l1_ratio': 0.4626599986704694}. Best is trial 0 with value: 0.5935639212572479.
Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:45:53,069] Trial 26 finished with value: 0.5935639212572479 and parameters: {'l1_ratio': 0.3312072877416

Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5107296137339056
[I 2024-04-16 20:46:02,138] Trial 48 finished with value: 0.5844439566015824 and parameters: {'l1_ratio': 0.04703679633024381}. Best is trial 27 with value: 0.5944002475623498.
Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:46:02,618] Trial 49 finished with value: 0.5935639212572479 and parameters: {'l1_ratio': 0.22788096176860018}. Best is trial 27 with value: 0.5944002475623498.
Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5407725321888412
[I 2024-04-16 20:46:03,204] Trial 50 finished with value: 0.5927887274587983 and parameters: {'l1_ratio': 0.164133117

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2629880548774616
Fold 2 IBS: 0.22682971587463843
Fold 3 IBS: 0.2142744333587882
Fold 4 IBS: 0.327962657160629
Fold 5 IBS: 0.2513930257838356
[I 2024-04-16 20:46:24,998] Trial 0 finished with value: 0.25668957741107057 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.25668957741107057.
Fold 1 IBS: 0.26302002003329694
Fold 2 IBS: 0.22767668645196562
Fold 3 IBS: 0.21651169127564762
Fold 4 IBS: 0.3278383802449254
Fold 5 IBS: 0.251406650001525
[I 2024-04-16 20:46:25,396] Trial 1 finished with value: 0.25729068560147217 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.25668957741107057.
Fold 1 IBS: 0.2630228044087859
Fold 2 IBS: 0.22799841306229704
Fold 3 IBS: 0.21729913336128845
Fold 4 IBS: 0.32770092293308273
Fold 5 IBS: 0.25140262266410585
[I 2024-04-16 20:46:25,954] Trial 2 finished with value: 0.25748477928591196 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.25668957741107057.

Fold 4 IBS: 0.32788853340548396
Fold 5 IBS: 0.2514031546088697
[I 2024-04-16 20:46:37,886] Trial 25 finished with value: 0.2571456705541921 and parameters: {'l1_ratio': 0.3418550839715946}. Best is trial 22 with value: 0.23533397914646725.
Fold 1 IBS: 0.26303018415882823
Fold 2 IBS: 0.22806664574073263
Fold 3 IBS: 0.21746526829451152
Fold 4 IBS: 0.32780607199164796
Fold 5 IBS: 0.25141701260574734
[I 2024-04-16 20:46:38,263] Trial 26 finished with value: 0.25755703655829354 and parameters: {'l1_ratio': 0.2166744240378211}. Best is trial 22 with value: 0.23533397914646725.
Fold 1 IBS: 0.24782330213255813
Fold 2 IBS: 0.23007189637973702
Fold 3 IBS: 0.22262771839375373
Fold 4 IBS: 0.24555328608314733
Fold 5 IBS: 0.23056099091292143
[I 2024-04-16 20:46:38,596] Trial 27 finished with value: 0.23532743878042353 and parameters: {'l1_ratio': 0.0607792618006927}. Best is trial 27 with value: 0.23532743878042353.
Fold 1 IBS: 0.24805481376325186
Fold 2 IBS: 0.22959547812194095
Fold 3 IBS: 0.221312

Fold 4 IBS: 0.32797944019122877
Fold 5 IBS: 0.25138371061424253
[I 2024-04-16 20:46:47,646] Trial 50 finished with value: 0.2567766348124616 and parameters: {'l1_ratio': 0.5938078108685914}. Best is trial 31 with value: 0.235303997907001.
Fold 1 IBS: 0.24768341105530375
Fold 2 IBS: 0.23041707644787301
Fold 3 IBS: 0.2236249361762146
Fold 4 IBS: 0.24481640722812334
Fold 5 IBS: 0.23035582265336363
[I 2024-04-16 20:46:47,951] Trial 51 finished with value: 0.23537953071217568 and parameters: {'l1_ratio': 0.04688889313970372}. Best is trial 31 with value: 0.235303997907001.
Fold 1 IBS: 0.24806550919882117
Fold 2 IBS: 0.22957573980711704
Fold 3 IBS: 0.22125911791840489
Fold 4 IBS: 0.2467250479679731
Fold 5 IBS: 0.2514277712699898
[I 2024-04-16 20:46:48,245] Trial 52 finished with value: 0.2394106372324612 and parameters: {'l1_ratio': 0.08468086971959793}. Best is trial 31 with value: 0.235303997907001.
Fold 1 IBS: 0.24774714762624245
Fold 2 IBS: 0.23025334582028947
Fold 3 IBS: 0.2231468863775

Fold 5 IBS: 0.23079802205404368
[I 2024-04-16 20:46:58,476] Trial 75 finished with value: 0.23529785796826858 and parameters: {'l1_ratio': 0.07963322397579652}. Best is trial 75 with value: 0.23529785796826858.
Fold 1 IBS: 0.26305036827463507
Fold 2 IBS: 0.2292795705291077
Fold 3 IBS: 0.22047917914446627
Fold 4 IBS: 0.24750047636825534
Fold 5 IBS: 0.2514171371521248
[I 2024-04-16 20:46:58,783] Trial 76 finished with value: 0.24234534629371782 and parameters: {'l1_ratio': 0.1018844983216041}. Best is trial 75 with value: 0.23529785796826858.
Fold 1 IBS: 0.24752688572810783
Fold 2 IBS: 0.23087674258361573
Fold 3 IBS: 0.22502055681147848
Fold 4 IBS: 0.24391774622739776
Fold 5 IBS: 0.23008234793739007
[I 2024-04-16 20:46:59,107] Trial 77 finished with value: 0.23548485585759799 and parameters: {'l1_ratio': 0.030997576498634}. Best is trial 75 with value: 0.23529785796826858.
Fold 1 IBS: 0.2630449068239805
Fold 2 IBS: 0.22889032661881023
Fold 3 IBS: 0.2194896832312748
Fold 4 IBS: 0.32772707

In [88]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [89]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.594
train_ibs:  0.235


#### Test

In [90]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [91]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.09486155085202524)

test_cindex : 0.568


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.07963322397579652)

test_ibs:  0.227


In [92]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [93]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 20:47:07,927] A new study created in memory with name: no-name-209d5f97-9322-4d32-bef7-20e26ed0b162


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.46215139442231074
Fold 2 C-index: 0.6434108527131783
Fold 3 C-index: 0.7574468085106383
Fold 4 C-index: 0.4448669201520912
Fold 5 C-index: 0.49356223175965663
[I 2024-04-16 20:47:12,932] Trial 0 finished with value: 0.5602876415115751 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.5602876415115751.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.6550387596899225
Fold 3 C-index: 0.774468085106383
Fold 4 C-index: 0.4144486692015209
Fold 5 C-index: 0.4678111587982833
[I 2024-04-16 20:47:16,627] Trial 1 finished with value: 0.5468154859536443 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717

Fold 3 C-index: 0.8638297872340426
Fold 4 C-index: 0.6197718631178707
Fold 5 C-index: 0.5622317596566524
[I 2024-04-16 20:47:51,563] Trial 15 finished with value: 0.6480653508920432 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 1, 'n_estimators': 170, 'oob_score': True, 'max_samples': 0.8107341403225247, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16866630138034366, 'warm_start': True}. Best is trial 15 with value: 0.6480653508920432.
Fold 1 C-index: 0.46215139442231074
Fold 2 C-index: 0.7441860465116279
Fold 3 C-index: 0.8553191489361702
Fold 4 C-index: 0.6159695817490495
Fold 5 C-index: 0.5836909871244635
[I 2024-04-16 20:47:52,791] Trial 16 finished with value: 0.6522634317487244 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 1, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.8029947290400248, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16939069998793

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:48:04,253] Trial 30 finished with value: 0.5 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 6, 'max_depth': 14, 'n_estimators': 78, 'oob_score': True, 'max_samples': 0.29592576068651116, 'max_features': None, 'min_weight_fraction_leaf': 0.2486312042554125, 'warm_start': True}. Best is trial 26 with value: 0.6863803462306775.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:48:05,046] Trial 31 finished with value: 0.5 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 154, 'oob_score': True, 'max_samples': 0.18146034387964277, 'max_features': None, 'min_weight_fraction_leaf': 0.12332617972097551, 'warm_start': True}. Best is trial 26 with value: 0.6863803462306775.
Fold 1 C-index: 0.46215139442231074
Fo

...

Fold 1 IBS: 0.246386460438482
Fold 2 IBS: 0.23218899742898796
Fold 3 IBS: 0.22946758410602397
Fold 4 IBS: 0.24118772219117468
Fold 5 IBS: 0.2303071920630508
[I 2024-04-16 20:50:58,547] Trial 32 finished with value: 0.2359075912455439 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 4, 'max_depth': 14, 'n_estimators': 175, 'oob_score': True, 'max_samples': 0.20444340416617043, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19558060234973634}. Best is trial 27 with value: 0.23583547084615547.
Fold 1 IBS: 0.24684478929600995
Fold 2 IBS: 0.2322329224950302
Fold 3 IBS: 0.23005427140610996
Fold 4 IBS: 0.2412308304074783
Fold 5 IBS: 0.2301271605653773
[I 2024-04-16 20:51:01,553] Trial 33 finished with value: 0.23609799483400112 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 235, 'oob_score': True, 'max_samples': 0.23980153995474307, 'max_features': 'log2', 'min_weight_fraction_leaf

Fold 5 IBS: 0.22978676956550373
[I 2024-04-16 20:51:58,964] Trial 47 finished with value: 0.23575678560731528 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 160, 'oob_score': True, 'max_samples': 0.13434415907111946, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.10612603041708193}. Best is trial 46 with value: 0.23574917305158732.
Fold 1 IBS: 0.24757897353687755
Fold 2 IBS: 0.2320725752850252
Fold 3 IBS: 0.2299544780683508
Fold 4 IBS: 0.24154478404388094
Fold 5 IBS: 0.23016145399352744
[I 2024-04-16 20:52:00,714] Trial 48 finished with value: 0.2362624529855324 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 18, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 86, 'oob_score': True, 'max_samples': 0.12964207163685623, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.10203918387448863}. Best is trial 46 with value: 0.23574917305158732.
Fold 1 IBS: 0.24671151151924972
Fold 2 IBS: 0.2321

Fold 1 IBS: 0.24631928839139375
Fold 2 IBS: 0.23211332325251055
Fold 3 IBS: 0.22987340393724426
Fold 4 IBS: 0.24110794902474542
Fold 5 IBS: 0.2296571564179856
[I 2024-04-16 20:52:45,476] Trial 63 finished with value: 0.2358142242047759 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 8, 'n_estimators': 150, 'oob_score': True, 'max_samples': 0.10056089265752739, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.026471209866795242}. Best is trial 46 with value: 0.23574917305158732.
Fold 1 IBS: 0.24648722835656947
Fold 2 IBS: 0.23206092276817514
Fold 3 IBS: 0.22928811947466984
Fold 4 IBS: 0.24128731957110403
Fold 5 IBS: 0.22992351169818162
[I 2024-04-16 20:52:48,788] Trial 64 finished with value: 0.23580942037374003 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 4, 'max_depth': 8, 'n_estimators': 104, 'oob_score': True, 'max_samples': 0.17392096150604247, 'max_features': 'sqrt', 'min_weight_fractio

Fold 5 IBS: 0.2519163373802201
[I 2024-04-16 20:53:34,303] Trial 78 finished with value: 0.24811294829357572 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 171, 'oob_score': True, 'max_samples': 0.9147835512625551, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.13257772422273512}. Best is trial 46 with value: 0.23574917305158732.
Fold 1 IBS: 0.24630596960013934
Fold 2 IBS: 0.23202892604980208
Fold 3 IBS: 0.23058361904315297
Fold 4 IBS: 0.24129573617042252
Fold 5 IBS: 0.2301768553877985
[I 2024-04-16 20:53:35,358] Trial 79 finished with value: 0.2360782212502631 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 11, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 37, 'oob_score': True, 'max_samples': 0.2189744006895358, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.17152888523323856}. Best is trial 46 with value: 0.23574917305158732.
Fold 1 IBS: 0.24642825384745612
Fold 2 IBS: 0.2320

Fold 1 IBS: 0.24603980142682666
Fold 2 IBS: 0.23212293756683253
Fold 3 IBS: 0.22980177266109972
Fold 4 IBS: 0.24117019375096946
Fold 5 IBS: 0.23000323031974987
[I 2024-04-16 20:54:46,192] Trial 94 finished with value: 0.23582758714509566 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 9, 'min_samples_leaf': 4, 'max_depth': 4, 'n_estimators': 186, 'oob_score': True, 'max_samples': 0.15211446012016716, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.1336981908011637}. Best is trial 84 with value: 0.23572114547332967.
Fold 1 IBS: 0.24609144860992238
Fold 2 IBS: 0.23210462726436726
Fold 3 IBS: 0.23027168009436597
Fold 4 IBS: 0.24100241364516664
Fold 5 IBS: 0.2300638162943246
[I 2024-04-16 20:54:57,781] Trial 95 finished with value: 0.23590679718162938 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 11, 'min_samples_leaf': 13, 'max_depth': 3, 'n_estimators': 489, 'oob_score': True, 'max_samples': 0.12627955102410032, 'max_features': 'auto', 'min_weight_fraction

In [94]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [95]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.815
train_ibs:  0.236


#### Test

In [96]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [97]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=14, max_features='auto', max_leaf_nodes=19,
                     max_samples=0.9566855419974443, min_samples_leaf=1,
                     min_samples_split=7,
                     min_weight_fraction_leaf=0.017888337982888625,
                     n_estimators=295, random_state=123, warm_start=True)

test_cindex:  0.562


RandomSurvivalForest(max_depth=3, max_features='auto', max_leaf_nodes=11,
                     max_samples=0.1369774849788948, min_samples_leaf=2,
                     min_samples_split=20,
                     min_weight_fraction_leaf=0.07082006896114468,
                     n_estimators=186, oob_score=True, random_state=123)

test_ibs:  0.23


In [98]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [99]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [100]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:55:31,125] A new study created in memory with name: no-name-845c7c6b-c15f-4654-b7b9-f387db353881


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.8765957446808511
Fold 4 C-index: 0.6045627376425855
Fold 5 C-index: 0.6394849785407726
[I 2024-04-16 20:55:35,057] Trial 0 finished with value: 0.6582032466680394 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6582032466680394.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 20:55:41,665] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.43824701195219123
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.8170212765957446
Fold 4 C-index: 0.5817490494296578
Fold 5 C-index: 0.5536480686695279
[I 2024-04-16 20:56:22,530] Trial 15 finished with value: 0.6300710658255483 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.6893948377364146.
Fold 1 C-index: 0.43824701195219123
Fold 2 C-index: 0.6802325581395349
Fold 3 C-index: 0.7553191489361702
Fold 4 C-index: 0.5228136882129277
Fold 5 C-index: 0.5128755364806867
[I 2024-04-16 20:56:24,031] Trial 16 finished with value: 0.5818975887443021 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_fe

Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.7868217054263565
Fold 3 C-index: 0.8595744680851064
Fold 4 C-index: 0.5627376425855514
Fold 5 C-index: 0.5965665236051502
[I 2024-04-16 20:56:43,600] Trial 30 finished with value: 0.6456022193348552 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 12, 'min_samples_leaf': 4, 'max_depth': 7, 'n_estimators': 343, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.3838693489032564, 'min_weight_fraction_leaf': 0.03868084640768137}. Best is trial 12 with value: 0.6893948377364146.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.7441860465116279
Fold 3 C-index: 0.8382978723404255
Fold 4 C-index: 0.5817490494296578
Fold 5 C-index: 0.592274678111588
[I 2024-04-16 20:56:46,889] Trial 31 finished with value: 0.6357636806730822 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 12, 'n_estimators': 407, 'oob_score': False, 'warm_start': True, 'max_feat

...

Fold 1 IBS: 0.2466509090859412
Fold 2 IBS: 0.23214878301411424
Fold 3 IBS: 0.2293161545437167
Fold 4 IBS: 0.2416001168061908
Fold 5 IBS: 0.23029389935280453
[I 2024-04-16 21:02:42,825] Trial 30 finished with value: 0.2360019725605535 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 11, 'n_estimators': 176, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.6685390299837802, 'min_weight_fraction_leaf': 0.39443765410475895}. Best is trial 1 with value: 0.2359075685671228.
Fold 1 IBS: 0.24658437242309666
Fold 2 IBS: 0.23217269485057873
Fold 3 IBS: 0.22938349245574668
Fold 4 IBS: 0.24142526486365426
Fold 5 IBS: 0.23026356293423877
[I 2024-04-16 21:02:45,147] Trial 31 finished with value: 0.23596587750546302 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 10, 'min_samples_leaf': 12, 'max_depth': 8, 'n_estimators': 195, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.

Fold 1 IBS: 0.2503238480535292
Fold 2 IBS: 0.22832308515260305
Fold 3 IBS: 0.22312583056505117
Fold 4 IBS: 0.24757766108141138
Fold 5 IBS: 0.2298732183504811
[I 2024-04-16 21:03:06,085] Trial 45 finished with value: 0.23584472864061518 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 16, 'min_samples_leaf': 5, 'max_depth': 19, 'n_estimators': 75, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8505275653973254, 'min_weight_fraction_leaf': 0.010891498916606263}. Best is trial 37 with value: 0.2351681311429794.
Fold 1 IBS: 0.25423248317390557
Fold 2 IBS: 0.23009681280030897
Fold 3 IBS: 0.22359000801229995
Fold 4 IBS: 0.25495747346505554
Fold 5 IBS: 0.23097528293655295
[I 2024-04-16 21:03:06,864] Trial 46 finished with value: 0.23877041207762462 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 19, 'n_estimators': 27, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.86265779

Fold 1 IBS: 0.24861024845564955
Fold 2 IBS: 0.22628374640473373
Fold 3 IBS: 0.22304190204035124
Fold 4 IBS: 0.24895737324177963
Fold 5 IBS: 0.23206607842779056
[I 2024-04-16 21:03:24,275] Trial 60 finished with value: 0.23579186971406094 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 18, 'n_estimators': 85, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.5421651479480726, 'min_weight_fraction_leaf': 0.08280808152330867}. Best is trial 37 with value: 0.2351681311429794.
Fold 1 IBS: 0.24827434177310634
Fold 2 IBS: 0.22497819527176266
Fold 3 IBS: 0.22107121877038577
Fold 4 IBS: 0.24897519780858499
Fold 5 IBS: 0.23171809515894556
[I 2024-04-16 21:03:25,369] Trial 61 finished with value: 0.23500340975655706 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 18, 'n_estimators': 90, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples':

Fold 1 IBS: 0.2476186048371288
Fold 2 IBS: 0.22872664517046198
Fold 3 IBS: 0.22689052239411525
Fold 4 IBS: 0.24591981570934932
Fold 5 IBS: 0.23215103509735893
[I 2024-04-16 21:03:41,797] Trial 75 finished with value: 0.23626132464168284 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 13, 'min_samples_leaf': 2, 'max_depth': 17, 'n_estimators': 82, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.546882037911445, 'min_weight_fraction_leaf': 0.15537264884677487}. Best is trial 61 with value: 0.23500340975655706.
Fold 1 IBS: 0.2484067040187848
Fold 2 IBS: 0.23025709083694096
Fold 3 IBS: 0.2273416756984175
Fold 4 IBS: 0.2444028493197838
Fold 5 IBS: 0.23289787879407822
[I 2024-04-16 21:03:44,973] Trial 76 finished with value: 0.23666123973360106 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 18, 'n_estimators': 331, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.68

Fold 1 IBS: 0.2472898739458269
Fold 2 IBS: 0.22726208935373643
Fold 3 IBS: 0.22289296673786196
Fold 4 IBS: 0.2504906341074561
Fold 5 IBS: 0.23197377846213324
[I 2024-04-16 21:03:51,058] Trial 90 finished with value: 0.23598186852140293 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 19, 'min_samples_leaf': 14, 'max_depth': 18, 'n_estimators': 3, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.873448690418433, 'min_weight_fraction_leaf': 0.05305541913345835}. Best is trial 81 with value: 0.23323924673335536.
Fold 1 IBS: 0.25088821247966814
Fold 2 IBS: 0.22829990283554244
Fold 3 IBS: 0.22583027016783427
Fold 4 IBS: 0.24669491907386742
Fold 5 IBS: 0.2315666548174323
[I 2024-04-16 21:03:51,568] Trial 91 finished with value: 0.2366559918748689 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 13, 'max_depth': 17, 'n_estimators': 30, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.

In [101]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [102]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.728
train_ibs:  0.233


#### Test

In [103]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [104]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=15, max_features='auto', max_leaf_nodes=10,
                   max_samples=0.8031091419587425, min_samples_leaf=2,
                   min_weight_fraction_leaf=0.011753369939040801,
                   n_estimators=473, random_state=123, warm_start=True)

C-index score: 0.59


ExtraSurvivalTrees(max_depth=17, max_features='log2', max_leaf_nodes=20,
                   max_samples=0.7941950203899084, min_samples_leaf=10,
                   min_samples_split=15,
                   min_weight_fraction_leaf=0.0647704116037696, n_estimators=6,
                   random_state=123, warm_start=True)

IBS: 0.228


In [105]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis


#### Train

In [106]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 21:03:55,714] A new study created in memory with name: no-name-7ed0dc64-d63d-4b39-9e1d-9472c41476f4


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:04:17,920] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:04:30,986] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:09:24,030] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.7784830438231676, 'learning_rate': 0.02439641670004065, 'dropout_rate': 0.4821375662037144, 'n_estimators': 148, 'criterion': 'friedman_mse', 'ccp_alpha': 6.245821408640139, 'min_weight_fraction_leaf': 0.3469021849356369, 'max_features': 1, 'min_impurity_decrease': 0.002901900339595834, 'validation_fraction': 0.40492470595401014, 'min_samples_split': 6, 'max_leaf_nodes': 15, 'min_samples_leaf': 12, 'max_depth': 5}. Best is trial 9 with value: 0.5532163203036308.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:10:05,635] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.5522283925781899, 'learning_rate': 0.010913227078192221, 'dropout_rate': 0.275468298707492, 'n_estimators': 392, 'criterion': 'squared_error', 'c

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:14:57,473] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8806073308147355, 'learning_rate': 0.022376976197093057, 'dropout_rate': 0.6881130985931054, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 2.7801390404743485, 'min_weight_fraction_leaf': 0.14826348238129222, 'max_features': 1, 'min_impurity_decrease': 3.2746646321179855e-06, 'validation_fraction': 0.654336976255422, 'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 14, 'max_depth': 5}. Best is trial 9 with value: 0.5532163203036308.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 21:16:57,478] Trial 27 finished with value: 0.5 and parameters: {'subsample': 0.6416570269538786, 'learning_rate': 0.08875752985571464, 'dropout_rate': 0.19627515205758927, 'n_estimators': 344, 'criterion': 'friedman_mse'

...

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 21:59:11,870] Trial 45 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6347814711411004, 'learning_rate': 0.024918730390022577, 'dropout_rate': 0.4346567206287487, 'n_estimators': 71, 'criterion': 'friedman_mse', 'ccp_alpha': 9.50847856451098, 'min_weight_fraction_leaf': 0.204461577134346, 'max_features': 'auto', 'min_impurity_decrease': 7.287535049264176e-05, 'validation_fraction': 0.296384154065476, 'min_samples_split': 10, 'max_leaf_nodes': 11, 'min_samples_leaf': 5, 'max_depth': 9}. Best is trial 3 with value: 0.2359278435123307.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 21:59:14,858] Trial 46 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8011995442645217, 'learni

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:00:52,448] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.2887765610954624, 'learning_rate': 0.09459281124929597, 'dropout_rate': 0.4973870333170236, 'n_estimators': 343, 'criterion': 'friedman_mse', 'ccp_alpha': 2.3090586149455574, 'min_weight_fraction_leaf': 0.16870893777683815, 'max_features': None, 'min_impurity_decrease': 9.555769670116001e-07, 'validation_fraction': 0.8832333189890804, 'min_samples_split': 13, 'max_leaf_nodes': 12, 'min_samples_leaf': 8, 'max_depth': 12}. Best is trial 3 with value: 0.2359278435123307.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 22:01:04,307] Trial 57 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7386655194472563, 'learning_rate': 0.08121117574322781, 'dropout_rate': 0.7097006096

Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:02:15,850] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6828521170655483, 'learning_rate': 0.022258646319799062, 'dropout_rate': 0.6010596795994926, 'n_estimators': 87, 'criterion': 'squared_error', 'ccp_alpha': 5.4909073156880295, 'min_weight_fraction_leaf': 0.1860826779096332, 'max_features': 'auto', 'min_impurity_decrease': 0.00019344127251504632, 'validation_fraction': 0.3679720810813184, 'min_samples_split': 9, 'max_leaf_nodes': 18, 'min_samples_leaf': 3, 'max_depth': 4}. Best is trial 3 with value: 0.2359278435123307.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:02:28,583] Trial 68 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.640231117985754, 'learning_rate': 0.0690549249219706, 'dropout_rate': 0.7519380649315037, 'n_estimators': 327, 'crit

Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:03:45,131] Trial 79 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.5239417471386933, 'learning_rate': 0.020324753076534783, 'dropout_rate': 0.25505231881333934, 'n_estimators': 219, 'criterion': 'squared_error', 'ccp_alpha': 8.06685235564664, 'min_weight_fraction_leaf': 0.10889477607680587, 'max_features': None, 'min_impurity_decrease': 2.113599021609697e-05, 'validation_fraction': 0.19421333161738485, 'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 3, 'max_depth': 10}. Best is trial 3 with value: 0.2359278435123307.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927107
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:04:00,704] Trial 80 finished with value: 0.2359278435123307 and parameters: {'sub

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:05:45,977] Trial 90 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7495496502041824, 'learning_rate': 0.0581528234960663, 'dropout_rate': 0.7348956640088382, 'n_estimators': 384, 'criterion': 'friedman_mse', 'ccp_alpha': 3.5249398640258107, 'min_weight_fraction_leaf': 0.3046819488123618, 'max_features': 'log2', 'min_impurity_decrease': 6.057529663822337e-06, 'validation_fraction': 0.2789829481077005, 'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 18, 'max_depth': 15}. Best is trial 3 with value: 0.2359278435123307.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 22:05:46,651] Trial 91 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.5619495945045616, 'learning_rate': 0.06700515143766485

In [107]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [108]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.553
train_ibs:  0.236


#### Test

In [109]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [110]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.07426378544613033,
                                 criterion='squared_error',
                                 dropout_rate=0.28125955124323376,
                                 learning_rate=0.013102111413618,
                                 max_features='auto', max_leaf_nodes=16,
                                 min_impurity_decrease=1.4994028685178666e-07,
                                 min_samples_leaf=10,
                                 min_weight_fraction_leaf=0.27579636299120275,
                                 n_estimators=406, random_state=123,
                                 subsample=0.6059965408578151,
                                 validation_fraction=0.6359003593513561)

C-index score: 0.573


GradientBoostingSurvivalAnalysis(ccp_alpha=7.636828414433382,
                                 dropout_rate=0.6624131518860399,
                                 learning_rate=0.059007718703659076,
                                 max_depth=14, max_leaf_nodes=5,
                                 min_impurity_decrease=1.2496121527731156e-07,
                                 min_samples_leaf=4, min_samples_split=12,
                                 min_weight_fraction_leaf=0.121833187268437,
                                 n_estimators=338, random_state=123,
                                 subsample=0.7023824046660451,
                                 validation_fraction=0.5944318794450425)

IBS: 0.229


In [111]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [112]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [113]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 22:07:17,820] A new study created in memory with name: no-name-ae39b9de-406b-4e5e-b47d-62e2ba553934


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.6085271317829457
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5579399141630901
[I 2024-04-16 22:07:18,740] Trial 0 finished with value: 0.5743544316635039 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5743544316635039.
Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.40304182509505704
Fold 5 C-index: 0.5622317596566524
[I 2024-04-16 22:07:25,787] Trial 1 finished with value: 0.593817451925007 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.593817451925007.
Fold 1 C-index: 0.4940239043824701
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8085106382978723
Fol

Fold 1 C-index: 0.4940239043824701
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.38783269961977185
Fold 5 C-index: 0.5793991416309013
[I 2024-04-16 22:08:21,595] Trial 19 finished with value: 0.5934881605071334 and parameters: {'subsample': 0.3662035355915435, 'dropout_rate': 0.7186486309433362, 'n_estimators': 131, 'learning_rate': 0.001397855219367588}. Best is trial 10 with value: 0.5989530559794313.
Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.689922480620155
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.3916349809885932
Fold 5 C-index: 0.5793991416309013
[I 2024-04-16 22:08:24,080] Trial 20 finished with value: 0.5943461057627897 and parameters: {'subsample': 0.19810816919434093, 'dropout_rate': 0.18129322567164805, 'n_estimators': 245, 'learning_rate': 0.02599747631629869}. Best is trial 10 with value: 0.5989530559794313.
Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.6937984496124031
Fold 3 C-index: 0.8212765957446

Fold 5 C-index: 0.575107296137339
[I 2024-04-16 22:09:53,998] Trial 37 finished with value: 0.5975370646695548 and parameters: {'subsample': 0.21856492743694161, 'dropout_rate': 0.40697127631393737, 'n_estimators': 478, 'learning_rate': 0.0942551407730424}. Best is trial 36 with value: 0.6006225515087767.
Fold 1 C-index: 0.4940239043824701
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.38783269961977185
Fold 5 C-index: 0.5708154506437768
[I 2024-04-16 22:10:02,782] Trial 38 finished with value: 0.5925466161081581 and parameters: {'subsample': 0.8179022915704841, 'dropout_rate': 0.1537716325192168, 'n_estimators': 497, 'learning_rate': 0.0802388198308747}. Best is trial 36 with value: 0.6006225515087767.
Fold 1 C-index: 0.4940239043824701
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.8085106382978723
Fold 4 C-index: 0.38783269961977185
Fold 5 C-index: 0.575107296137339
[I 2024-04-16 22:10:03,939] Trial 39 finished with value: 0.59495537280

...

Fold 2 IBS: 0.24471484981206823
Fold 3 IBS: 0.1838567017320815
Fold 4 IBS: 0.3810912888510208
Fold 5 IBS: 0.2589104839271485
[I 2024-04-16 22:14:58,218] Trial 19 finished with value: 0.27985526623409424 and parameters: {'subsample': 0.650899739648275, 'dropout_rate': 0.9954484919483726, 'n_estimators': 400, 'learning_rate': 0.011730584939170362}. Best is trial 13 with value: 0.23592875599808352.
Fold 1 IBS: 0.26865035103700036
Fold 2 IBS: 0.2248930841263748
Fold 3 IBS: 0.20869332419609296
Fold 4 IBS: 0.2874440317119254
Fold 5 IBS: 0.23152896013842253
[I 2024-04-16 22:14:58,704] Trial 20 finished with value: 0.2442419502419632 and parameters: {'subsample': 0.9010126869682862, 'dropout_rate': 0.6173032374045795, 'n_estimators': 67, 'learning_rate': 0.0257290746252228}. Best is trial 13 with value: 0.23592875599808352.
Fold 1 IBS: 0.24734112833834176
Fold 2 IBS: 0.2318949440751368
Fold 3 IBS: 0.22874717671047945
Fold 4 IBS: 0.24231017005455716
Fold 5 IBS: 0.22934262647544185
[I 2024-04-16

Fold 4 IBS: 0.36238164279418583
Fold 5 IBS: 0.24860764869709162
[I 2024-04-16 22:15:10,399] Trial 38 finished with value: 0.2711478905472867 and parameters: {'subsample': 0.822360889083144, 'dropout_rate': 0.13272164755980653, 'n_estimators': 84, 'learning_rate': 0.060691705645777076}. Best is trial 21 with value: 0.2359272091307914.
Fold 1 IBS: 0.3201955917664464
Fold 2 IBS: 0.2341771425521463
Fold 3 IBS: 0.1886473281684259
Fold 4 IBS: 0.3660097798438394
Fold 5 IBS: 0.25014108036086385
[I 2024-04-16 22:15:11,497] Trial 39 finished with value: 0.2718341845383444 and parameters: {'subsample': 0.3972684667616401, 'dropout_rate': 0.8833943811644197, 'n_estimators': 159, 'learning_rate': 0.027825288019738728}. Best is trial 21 with value: 0.2359272091307914.
Fold 1 IBS: 0.28010588003020465
Fold 2 IBS: 0.22666590260107977
Fold 3 IBS: 0.2019086955339845
Fold 4 IBS: 0.30841973010009777
Fold 5 IBS: 0.23462207425601805
[I 2024-04-16 22:15:11,927] Trial 40 finished with value: 0.2503444565042769

Fold 1 IBS: 0.24890979703840563
Fold 2 IBS: 0.23003148247286287
Fold 3 IBS: 0.2254557529769063
Fold 4 IBS: 0.24728830711327457
Fold 5 IBS: 0.228782911842937
[I 2024-04-16 22:15:28,163] Trial 58 finished with value: 0.23609365028887724 and parameters: {'subsample': 0.7920490567025045, 'dropout_rate': 0.8710743587053082, 'n_estimators': 17, 'learning_rate': 0.01472286899228218}. Best is trial 21 with value: 0.2359272091307914.
Fold 1 IBS: 0.33137168930730976
Fold 2 IBS: 0.2439619426483204
Fold 3 IBS: 0.18985013201621057
Fold 4 IBS: 0.377785990019265
Fold 5 IBS: 0.25780738622080585
[I 2024-04-16 22:15:28,630] Trial 59 finished with value: 0.2801554280423823 and parameters: {'subsample': 0.4281699397350103, 'dropout_rate': 0.7082113893699592, 'n_estimators': 62, 'learning_rate': 0.08322655251020356}. Best is trial 21 with value: 0.2359272091307914.
Fold 1 IBS: 0.24823638249156682
Fold 2 IBS: 0.23073253702170793
Fold 3 IBS: 0.22695457812129097
Fold 4 IBS: 0.2451310203758235
Fold 5 IBS: 0.22

Fold 4 IBS: 0.3039939279331561
Fold 5 IBS: 0.23528769396515306
[I 2024-04-16 22:15:44,494] Trial 77 finished with value: 0.24953894822442893 and parameters: {'subsample': 0.8256210781433221, 'dropout_rate': 0.77514909005388, 'n_estimators': 74, 'learning_rate': 0.03072890021688105}. Best is trial 69 with value: 0.2359207274102837.
Fold 1 IBS: 0.3138835900465689
Fold 2 IBS: 0.23776002911750155
Fold 3 IBS: 0.19108475850581674
Fold 4 IBS: 0.3571181386469282
Fold 5 IBS: 0.25096187498366274
[I 2024-04-16 22:15:44,986] Trial 78 finished with value: 0.27016167826009563 and parameters: {'subsample': 0.8981621675854204, 'dropout_rate': 0.8182308606189214, 'n_estimators': 57, 'learning_rate': 0.06751738229506911}. Best is trial 69 with value: 0.2359207274102837.
Fold 1 IBS: 0.249960989092115
Fold 2 IBS: 0.22940983791440717
Fold 3 IBS: 0.22444822802920752
Fold 4 IBS: 0.25088958787309706
Fold 5 IBS: 0.2285465576559458
[I 2024-04-16 22:15:45,278] Trial 79 finished with value: 0.2366510401129545 and

Fold 1 IBS: 0.24741169763879048
Fold 2 IBS: 0.2317895664300078
Fold 3 IBS: 0.22858459606339743
Fold 4 IBS: 0.24255845922196848
Fold 5 IBS: 0.2293051926134131
[I 2024-04-16 22:15:52,792] Trial 97 finished with value: 0.23592990239351544 and parameters: {'subsample': 0.93525187912044, 'dropout_rate': 0.5247006975414128, 'n_estimators': 9, 'learning_rate': 0.003396199225455108}. Best is trial 81 with value: 0.2359201657784371.
Fold 1 IBS: 0.24730327400164734
Fold 2 IBS: 0.23196284948715093
Fold 3 IBS: 0.22885037550107723
Fold 4 IBS: 0.24215795642981985
Fold 5 IBS: 0.2293664065113921
[I 2024-04-16 22:15:53,089] Trial 98 finished with value: 0.2359281723862175 and parameters: {'subsample': 0.8443630472097454, 'dropout_rate': 0.45029073287468846, 'n_estimators': 1, 'learning_rate': 0.009810701317067457}. Best is trial 81 with value: 0.2359201657784371.
Fold 1 IBS: 0.24760599763754257
Fold 2 IBS: 0.23149495327850178
Fold 3 IBS: 0.2282010641579017
Fold 4 IBS: 0.2431802635319715
Fold 5 IBS: 0.2

In [114]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [115]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.602
train_ibs:  0.236


#### Test

In [116]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [117]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3678471747462695,
                                              learning_rate=0.039763754907327475,
                                              n_estimators=349,
                                              random_state=123,
                                              subsample=0.14305853399792298)

C-index score: 0.567


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7120491271209716,
                                              learning_rate=0.022969066082900873,
                                              n_estimators=1, random_state=123,
                                              subsample=0.9544791688081666)

IBS: 0.229


In [118]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [119]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.815,1.0
ExtraSurvivalTrees,0.728,2.0
ComponentwiseGradientBoosting,0.602,3.0
CoxLasso,0.594,4.5
CoxElastic,0.594,4.5
CoxRidge,0.585,6.0
CoxPH,0.580,7.0
GradientBoosting,0.553,8.0


In [120]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.233,1.0
CoxElastic,0.235,2.0
CoxRidge,0.236,4.5
Randomsurvivalforest,0.236,4.5
GradientBoosting,0.236,4.5
ComponentwiseGradientBoosting,0.236,4.5
CoxPH,0.250,7.0
CoxLasso,0.257,8.0


In [121]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.590,1.0
GradientBoosting,0.573,2.0
CoxRidge,0.569,3.0
CoxElastic,0.568,4.0
ComponentwiseGradientBoosting,0.567,5.0
Randomsurvivalforest,0.562,6.0
CoxPH,0.553,7.5
CoxLasso,0.553,7.5


In [122]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
CoxElastic,0.227,1.0
ExtraSurvivalTrees,0.228,2.0
CoxRidge,0.229,4.0
GradientBoosting,0.229,4.0
ComponentwiseGradientBoosting,0.229,4.0
Randomsurvivalforest,0.230,6.0
CoxPH,0.246,7.5
CoxLasso,0.246,7.5


In [123]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/dfs/yeojohnson/plsr/'  # Folder path where you want to save the files

# Modify the file names to match the desired format
modified_file_names = ['d3_dfs_yeojohnson_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [124]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-16
